
* [#68](https://github.com/salgo60/SCB-Wikidata/issues/68) 
* Notebook [lansstyrelsen_68.ipynb](https://github.com/salgo60/SCB-Wikidata/blob/main/notebook/lansstyrelsen_68.ipynb)




In [1]:
import time

from datetime import datetime

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Start: 2026-02-17 08:51:24


In [2]:

SCRIPT_NAME = "lansstyrelsen_68.ipynb"
SCRIPT_URL = (
    "https://github.com/salgo60/SCB-Wikidata/"
    "blob/master/notebook/lansstyrelsen_68.ipynb"
) 


In [3]:
def read_domains(file_path):
    print(f"[DEBUG] Reading domains from: {file_path}")
    df = pd.read_csv(file_path, header=0)   # <- skip header row
    domains_list = df.iloc[:, 0].dropna().unique().tolist()
    print(f"[DEBUG] Found {len(domains_list)} domains.")
    return domains_list


In [4]:
import requests

def fetch_sitematrix_df():
    url = "https://meta.wikimedia.org/w/api.php"
    params = {
        "action": "sitematrix",
        "format": "json"
    }
    headers = {
        "User-Agent": "salgo60-language-fetcher/1.0 (salgo60@msn.com)"
    }

    print("[DEBUG] Fetching sitematrix…")
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    if "application/json" not in r.headers.get("Content-Type", ""):
        raise ValueError("Server returned non-JSON response")

    data = r.json()["sitematrix"]

    rows = []

    # --- language-specific sites ---
    for key, lang_block in data.items():
        if not key.isdigit():
            continue  # skip "count", "specials"

        lang_code = lang_block.get("code")
        lang_name = lang_block.get("name")

        for site in lang_block.get("site", []):
            rows.append({
                "lang_code": lang_code,
                "lang_name": lang_name,
                "project": site.get("project"),
                "url": site.get("url"),
                "dbname": site.get("dbname"),
                "site_name": site.get("sitename"),
                "closed": site.get("closed", False)
            })

    # --- special wikis (Wikidata, Commons, Meta, etc.) ---
    for site in data.get("specials", []):
        rows.append({
            "lang_code": "special",
            "lang_name": "special",
            "project": site.get("project"),
            "url": site.get("url"),
            "dbname": site.get("dbname"),
            "site_name": site.get("sitename"),
            "closed": site.get("closed", False)
        })

    return pd.DataFrame(rows)


In [5]:
import requests
import pandas as pd


HEADERS = {
    "User-Agent": "salgo60-language-fetcher/2.0 (https://github.com/salgo60) salgo60@msn.com"
}


df_lang_fetch = fetch_sitematrix_df()
df_lang_fetch["closed"] = df_lang_fetch["closed"].fillna(False).astype(bool)

df_lang_wikipedia = df_lang_fetch[
    (df_lang_fetch["site_name"] == "Wikipedia") &
    (
        (df_lang_fetch["lang_name"].str.lower() != "special") |
        (df_lang_fetch["dbname"] == "wikidatawiki")
    )
]  

df_lang_wikipedia.info()


[DEBUG] Fetching sitematrix…
<class 'pandas.core.frame.DataFrame'>
Index: 186 entries, 0 to 1047
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lang_code  186 non-null    object
 1   lang_name  185 non-null    object
 2   project    0 non-null      object
 3   url        186 non-null    object
 4   dbname     186 non-null    object
 5   site_name  186 non-null    object
 6   closed     186 non-null    bool  
dtypes: bool(1), object(6)
memory usage: 10.4+ KB


In [6]:
import os

# Get the current working directory
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)



Current Working Directory: /Users/salgo/Documents/GitHub/SCB-Wikidata/notebook


In [7]:
import os
import time
import random
import requests
import pandas as pd
from urllib.parse import urlparse
from tqdm.notebook import tqdm
file_path_domain = "sources/domains_ lansstyrelsen.csv"
domains = read_domains(file_path_domain)
print(domains)


[DEBUG] Reading domains from: sources/domains_ lansstyrelsen.csv
[DEBUG] Found 2 domains.
['lansstyrelsen.se', 'lst.se']


In [8]:
def resolve_api_base(lang):
    if lang == "special":
        # Wikidata (only valid "special" case in your pipeline)
        return "https://www.wikidata.org/w/api.php"

    return f"https://{lang}.wikipedia.org/w/api.php"


In [9]:
def request_timeout(lang):
    return 100 if lang == "special" else 10

In [10]:
# -----------------------------------------------------------
# Fetch exturlusage entries for one lang/domain
# -----------------------------------------------------------
def fetch_exturlusage(lang, domain):
    #base = f"https://{lang}.wikipedia.org/w/api.php"
    base = resolve_api_base(lang)
    params = {
        "action": "query",
        "format": "json",
        "list": "exturlusage",
        "euquery": domain,
        "eulimit": "max"
    }
    while True:
        #r = session.get(base, params=params, timeout=10)
        r = session.get(base, params=params, timeout=request_timeout(lang))        
        try:
            data = r.json()
        except ValueError:
            print(f"[WARN] {lang}: JSON decode failed")
            break

        for item in data.get("query", {}).get("exturlusage", []):
            yield {
                "lang": lang,
                "page_title": item.get("title"),
                "url": item.get("url"),
                "wiki_link": f"https://{lang}.wikipedia.org/wiki/{item.get('title').replace(' ', '_')}"
            }

        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.3)

In [11]:

# -------------------------
# Session & helpers
# -------------------------
session = requests.Session()
session.headers.update({"User-Agent": "SCB-LinkAudit/1.0 salgo60@msn.com"})

# we need some filtering  

print("Antal Språk:",len(df_lang_wikipedia ))
results = []
for _, row in df_lang_wikipedia.iterrows():
    lang = row["lang_code"]
    url  = row["url"]
    lang_name = row["lang_name"]
    before = len(results)
    #print(lang, url, lang_name,domains)
    for domain in domains:
        for entry in fetch_exturlusage(lang, domain):
            results.append(entry)
    after = len(results) 
    links = after-before
    print(lang, url, lang_name," - ", links)
    


Antal Språk: 186
aa https://aa.wikipedia.org Qafár af  -  0
ace https://ace.wikipedia.org Acèh  -  0
af https://af.wikipedia.org Afrikaans  -  1
ak https://ak.wikipedia.org None  -  0
ami https://ami.wikipedia.org Pangcah  -  0
an https://an.wikipedia.org aragonés  -  0
ast https://ast.wikipedia.org asturianu  -  22
av https://av.wikipedia.org авар  -  0
avk https://avk.wikipedia.org Kotava  -  0
ay https://ay.wikipedia.org Aymar aru  -  0
bar https://bar.wikipedia.org Boarisch  -  13
bbc https://bbc.wikipedia.org Batak Toba  -  0
bcl https://bcl.wikipedia.org Bikol Central  -  0
bi https://bi.wikipedia.org Bislama  -  0
bm https://bm.wikipedia.org bamanankan  -  0
bo https://bo.wikipedia.org བོད་ཡིག  -  0
br https://br.wikipedia.org brezhoneg  -  0
bs https://bs.wikipedia.org bosanski  -  23
btm https://btm.wikipedia.org Batak Mandailing  -  0
bug https://bug.wikipedia.org Basa Ugi  -  0
bxr https://bxr.wikipedia.org буряад  -  0
cbk-zam https://cbk-zam.wikipedia.org Chavacano de Zamb

In [12]:
df_lst = pd.DataFrame(results)
df_lst.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59912 entries, 0 to 59911
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        59912 non-null  object
 1   page_title  59912 non-null  object
 2   url         59912 non-null  object
 3   wiki_link   59912 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB


In [13]:
import pandas as pd

# --- Stats ---
total_links = len(df_lst)
total_unique_links = df_lst['url'].nunique()
num_languages = df_lst['lang'].nunique()
langs_sorted = df_lst['lang'].value_counts()

print("Total links:", total_links)
print("Total unique links:", total_unique_links)
print("Number of languages:", num_languages)
print("\nLanguages with most links:")
print(langs_sorted.to_string())


Total links: 59912
Total unique links: 40092
Number of languages: 38

Languages with most links:
lang
sv         52900
en          2281
no           990
special      934
de           855
fi           838
da           237
nl           106
es           105
pl            91
nn            76
gl            69
it            47
vi            40
ro            37
zh            37
ja            26
sco           24
bs            23
eu            22
ast           22
sw            22
simple        21
nds           21
lmo           20
lld           20
id            15
bar           13
vec            5
tl             3
frr            3
ms             2
sq             2
lb             1
is             1
su             1
ha             1
af             1


In [14]:
df_lst.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59912 entries, 0 to 59911
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        59912 non-null  object
 1   page_title  59912 non-null  object
 2   url         59912 non-null  object
 3   wiki_link   59912 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB


In [15]:
# --- Stats ---
total_links = len(df_lst)
total_unique_links = df_lst["url"].nunique()
langs_with_hits = sorted(df_lst["lang"].unique())

num_languages_found = len(langs_with_hits)
num_languages_checked = len(df_lang_wikipedia)        # alla språk som genomsöktes
num_languages_found = df_lst['lang'].nunique()


In [16]:
num_languages_checked 

186

In [17]:
import requests
from requests.exceptions import RequestException
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import pandas as pd
import time
import threading

# ==========================
# Konfiguration
# ==========================

OUTPUT = "checkpoint_links_lst.parquet"
MAX_WORKERS = 12
REQUEST_TIMEOUT = 15
CHECKPOINT_EVERY = 1000

ROOT_CANONICAL = "https://www.lansstyrelsen.se"

SOFT_404_PHRASES = [
    "Page Not Found",
    "Sidan hittades inte",
    "felaktig adress",
    "kontakta registrator",
    "sidan finns inte",
    "den här sidan kan inte visas",
    "Hoppsan! Vi kunde tyvärr inte hitta sidan"
]

USER_AGENT = "LinkChecker/1.0 (research; salgo60@msn.com)"

# ==========================
# Session (thread-safe)
# ==========================

thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({"User-Agent": USER_AGENT})
        thread_local.session = s
    return thread_local.session


# ==========================
# Hjälpfunktioner
# ==========================

def norm(u: str) -> str:
    return u.rstrip("/").lower()


def looks_like_soft_404(response) -> bool:
    text = (response.text or "").lower()

    for phrase in SOFT_404_PHRASES:
        if phrase in text:
            return True

    if "<title>" in text:
        title = text.split("<title>", 1)[1].split("</title>", 1)[0]
        if "404" in title.lower():
            return True
        if "sidan finns inte" in title.lower():
            return True

    return False



# ==========================
# URL-check
# ==========================
from urllib.parse import urljoin
from requests.exceptions import RequestException, TooManyRedirects

MAX_REDIRECTS = 10


def check_url(url: str) -> dict:
    session = get_session()

    visited = set()
    current_url = url

    try:
        for _ in range(MAX_REDIRECTS):

            # Skydd mot redirect-loop
            if current_url in visited:
                return {
                    "url": url,
                    "status": "error",
                    "reason": "redirect_loop",
                    "final_url": current_url,
                }

            visited.add(current_url)

            try:
                r = session.get(
                    current_url,
                    allow_redirects=False,  # vi hanterar redirect själva
                    timeout=REQUEST_TIMEOUT,
                    stream=True,
                )
            except RequestException as e:
                return {
                    "url": url,
                    "status": "error",
                    "reason": str(e),
                    "final_url": current_url,
                }

            status_code = r.status_code

            # ==========================
            # Redirect
            # ==========================
            if 300 <= status_code < 400:

                raw_location = r.headers.get("Location")
                r.close()

                if not raw_location:
                    return {
                        "url": url,
                        "status": "dead",
                        "reason": f"redirect_without_location ({status_code})",
                        "final_url": current_url,
                    }

                # Reparera trasig latin1-header
                try:
                    location = raw_location.encode("latin1").decode("utf-8")
                except Exception:
                    location = raw_location

                current_url = urljoin(current_url, location)
                continue

            # ==========================
            # Hard 404/500
            # ==========================
            if status_code >= 400:
                r.close()
                return {
                    "url": url,
                    "status": "dead",
                    "reason": f"HTTP {status_code}",
                    "final_url": current_url,
                }

            # ==========================
            # OK – kontrollera soft 404
            # ==========================
            try:
                content = r.text  # laddar body först nu
            except Exception:
                content = ""

            final_url = r.url
            r.close()

            # Redirect till root
            if norm(final_url) == norm(ROOT_CANONICAL) and norm(url) != norm(final_url):
                return {
                    "url": url,
                    "status": "dead",
                    "reason": "redirect_to_root",
                    "final_url": final_url,
                }

            if looks_like_soft_404(type("obj", (), {"text": content})()):
                return {
                    "url": url,
                    "status": "dead",
                    "reason": "soft_404",
                    "final_url": final_url,
                }

            return {
                "url": url,
                "status": "ok",
                "final_url": final_url,
            }

        # För många redirects
        return {
            "url": url,
            "status": "error",
            "reason": "too_many_redirects",
            "final_url": current_url,
        }

    except TooManyRedirects:
        return {
            "url": url,
            "status": "error",
            "reason": "too_many_redirects_exception",
        }

    except Exception as e:
        return {
            "url": url,
            "status": "error",
            "reason": f"unexpected_error: {str(e)}",
        }

# ==========================
# Internet Archive
# ==========================

def check_internet_archive(url: str) -> str | None:
    session = get_session()
    api = "https://archive.org/wayback/available"

    try:
        r = session.get(
            api,
            params={"url": url},
            timeout=10,
        )
        data = r.json()
    except Exception:
        return None

    snap = data.get("archived_snapshots", {}).get("closest")
    if snap and snap.get("available"):
        return snap.get("url")

    return None


# ==========================
# Worker
# ==========================

def process_url(url: str) -> dict:
    result = check_url(url)

    if result["status"] == "dead":
        ia_url = check_internet_archive(url)
        result["ia_url"] = ia_url
        result["ia_status"] = "available" if ia_url else "missing"
    else:
        result["ia_url"] = None
        result["ia_status"] = "skipped"

    return result


# ==========================
# Main
# ==========================

def run(df_lst: pd.DataFrame) -> pd.DataFrame:
    urls = [
        u for u in df_lst["url"].dropna().astype(str).unique()
    ]

    results = []
    start = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_url, u): u for u in urls}

        for i, future in enumerate(
            tqdm(as_completed(futures), total=len(futures), unit="link"),
            start=1
        ):
            result = future.result()
            results.append(result)

            if i % CHECKPOINT_EVERY == 0:
                pd.DataFrame(results).to_parquet(OUTPUT)

    df = pd.DataFrame(results)
    df.to_parquet(OUTPUT)

    elapsed = time.time() - start
    print(f"Klar på {elapsed/3600:.1f} timmar")

    return df


In [18]:
df_lst


,lang,page_title,url,wiki_link
0,af,Vändåtberget,http://www.lansstyrelsen.se/vasternorrland/Sv/...,https://af.wikipedia.org/wiki/Vändåtberget
1,ast,Provincia de Gävleborg,http://www.lansstyrelsen.se/gavleborg/,https://ast.wikipedia.org/wiki/Provincia_de_Gä...
2,ast,Provincia de Gävleborg,http://www.lansstyrelsen.se/gavleborg/Sv/Pages...,https://ast.wikipedia.org/wiki/Provincia_de_Gä...
3,ast,Provincia de Norrbotten,https://www.lansstyrelsen.se/norrbotten/,https://ast.wikipedia.org/wiki/Provincia_de_No...
4,ast,Provincia de Västernorrland,https://www.lansstyrelsen.se/vasternorrland/,https://ast.wikipedia.org/wiki/Provincia_de_Vä...
...,...,...,...,...
59907,special,Q79025739,http://www.y.lst.se/publikationerkungorelser/f...,https://special.wikipedia.org/wiki/Q79025739
59908,special,Q79025770,http://www.o.lst.se/,https://special.wikipedia.org/wiki/Q79025770
59909,special,Q79013220,http://www.ab.lst.se/,https://special.wikipedia.org/wiki/Q79013220
59910,special,Q79015737,http://www.ac.lst.se/files/yCiiQQQh.pdf,https://special.wikipedia.org/wiki/Q79015737


In [19]:
df_result = run(df_lst)


 27%|████████▉                        | 10916/40092 [17:10<1:10:42,  6.88link/s]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [20]:
df_result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40092 entries, 0 to 40091
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   url        40092 non-null  object
 1   status     40092 non-null  object
 2   final_url  40091 non-null  object
 3   ia_url     783 non-null    object
 4   ia_status  40092 non-null  object
 5   reason     2981 non-null   object
dtypes: object(6)
memory usage: 1.8+ MB


In [21]:
df_lst.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59912 entries, 0 to 59911
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        59912 non-null  object
 1   page_title  59912 non-null  object
 2   url         59912 non-null  object
 3   wiki_link   59912 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB


In [22]:

print("len(results):", len(results))

print("df_result.shape:", df_result.shape)
print(df_result.head(2))
print(df_result.tail(2))


len(results): 59912
df_result.shape: (40092, 6)
                                            url status  \
0  https://www.lansstyrelsen.se/vasternorrland/     ok   
1      https://www.lansstyrelsen.se/norrbotten/     ok   

                                      final_url ia_url ia_status reason  
0  https://www.lansstyrelsen.se/vasternorrland/   None   skipped    NaN  
1      https://www.lansstyrelsen.se/norrbotten/   None   skipped    NaN  
                                                     url status  \
40090  https://www.lansstyrelsen.se/download/18.1b1d3...     ok   
40091  https://www.lansstyrelsen.se/download/18.8cd5a...     ok   

                                               final_url ia_url ia_status  \
40090  https://www.lansstyrelsen.se/download/18.1b1d3...   None   skipped   
40091  https://www.lansstyrelsen.se/download/18.8cd5a...   None   skipped   

      reason  
40090    NaN  
40091    NaN  


In [23]:
df_result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40092 entries, 0 to 40091
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   url        40092 non-null  object
 1   status     40092 non-null  object
 2   final_url  40091 non-null  object
 3   ia_url     783 non-null    object
 4   ia_status  40092 non-null  object
 5   reason     2981 non-null   object
dtypes: object(6)
memory usage: 1.8+ MB


In [24]:
(df_result["status"] == "dead").mean()

0.0659483188666068

In [25]:
print("len(result):", len(results))
print("first:", results[0])
print("last:", results[-1])

len(result): 59912
first: {'lang': 'af', 'page_title': 'Vändåtberget', 'url': 'http://www.lansstyrelsen.se/vasternorrland/Sv/djur-och-natur/skyddad-natur/naturreservat-i-vasternorrland/ornskoldsviks-kommun/vandatberget/Pages/default.aspx', 'wiki_link': 'https://af.wikipedia.org/wiki/Vändåtberget'}
last: {'lang': 'special', 'page_title': 'Wikidata:Database reports/Constraint violations/P856', 'url': 'http://www.ab.lst.se/', 'wiki_link': 'https://special.wikipedia.org/wiki/Wikidata:Database_reports/Constraint_violations/P856'}


In [26]:
results[:3]

[{'lang': 'af',
  'page_title': 'Vändåtberget',
  'url': 'http://www.lansstyrelsen.se/vasternorrland/Sv/djur-och-natur/skyddad-natur/naturreservat-i-vasternorrland/ornskoldsviks-kommun/vandatberget/Pages/default.aspx',
  'wiki_link': 'https://af.wikipedia.org/wiki/Vändåtberget'},
 {'lang': 'ast',
  'page_title': 'Provincia de Gävleborg',
  'url': 'http://www.lansstyrelsen.se/gavleborg/',
  'wiki_link': 'https://ast.wikipedia.org/wiki/Provincia_de_Gävleborg'},
 {'lang': 'ast',
  'page_title': 'Provincia de Gävleborg',
  'url': 'http://www.lansstyrelsen.se/gavleborg/Sv/Pages/default.aspx',
  'wiki_link': 'https://ast.wikipedia.org/wiki/Provincia_de_Gävleborg'}]

In [27]:
df_result_dedup = df_result.drop_duplicates(subset="url", keep="last")

df_merged = df_lst.merge(
    df_result_dedup,
    on="url",
    how="left"
)



In [28]:
mask = df_merged["page_title"].str.contains("Naturrese", case=False, na=False)
df_naturreservat = df_merged[mask]

In [29]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59912 entries, 0 to 59911
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        59912 non-null  object
 1   page_title  59912 non-null  object
 2   url         59912 non-null  object
 3   wiki_link   59912 non-null  object
 4   status      59912 non-null  object
 5   final_url   59910 non-null  object
 6   ia_url      1193 non-null   object
 7   ia_status   59912 non-null  object
 8   reason      18854 non-null  object
dtypes: object(9)
memory usage: 4.1+ MB


In [30]:
df_naturreservat.head()


,lang,page_title,url,wiki_link,status,final_url,ia_url,ia_status,reason
70,da,Vålådalens naturreservat,http://www.lansstyrelsen.se/jamtland/Sv/djur-o...,https://da.wikipedia.org/wiki/Vålådalens_natur...,ok,https://www.lansstyrelsen.se/jamtland/besoksma...,None,skipped,NaN
102,da,Tärnö Naturreservat,http://www.lansstyrelsen.se/blekinge/Sv/djur-o...,https://da.wikipedia.org/wiki/Tärnö_Naturreservat,ok,https://www.lansstyrelsen.se/blekinge/besoksma...,None,skipped,NaN
139,da,Måkläppens naturreservat,http://www.lansstyrelsen.se/skane/Sv/djur-och-...,https://da.wikipedia.org/wiki/Måkläppens_natur...,ok,https://www.lansstyrelsen.se/skane/besoksmal/n...,None,skipped,NaN
140,da,Flommens naturreservat,http://www.lansstyrelsen.se/skane/Sv/djur-och-...,https://da.wikipedia.org/wiki/Flommens_naturre...,dead,https://www.lansstyrelsen.se/skane/Sv/djur-och...,None,missing,HTTP 404
187,da,Suseån (naturreservat),https://www.lansstyrelsen.se/halland/besoksmal...,https://da.wikipedia.org/wiki/Suseån_(naturres...,ok,https://www.lansstyrelsen.se/halland/besoksmal...,None,skipped,NaN


In [31]:
df_lst.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59912 entries, 0 to 59911
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        59912 non-null  object
 1   page_title  59912 non-null  object
 2   url         59912 non-null  object
 3   wiki_link   59912 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB


In [32]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59912 entries, 0 to 59911
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   lang        59912 non-null  object
 1   page_title  59912 non-null  object
 2   url         59912 non-null  object
 3   wiki_link   59912 non-null  object
 4   status      59912 non-null  object
 5   final_url   59910 non-null  object
 6   ia_url      1193 non-null   object
 7   ia_status   59912 non-null  object
 8   reason      18854 non-null  object
dtypes: object(9)
memory usage: 4.1+ MB


In [33]:
print("len(results):", len(results))
print("first:", results[0])
print("last:", results[-1])

len(results): 59912
first: {'lang': 'af', 'page_title': 'Vändåtberget', 'url': 'http://www.lansstyrelsen.se/vasternorrland/Sv/djur-och-natur/skyddad-natur/naturreservat-i-vasternorrland/ornskoldsviks-kommun/vandatberget/Pages/default.aspx', 'wiki_link': 'https://af.wikipedia.org/wiki/Vändåtberget'}
last: {'lang': 'special', 'page_title': 'Wikidata:Database reports/Constraint violations/P856', 'url': 'http://www.ab.lst.se/', 'wiki_link': 'https://special.wikipedia.org/wiki/Wikidata:Database_reports/Constraint_violations/P856'}


In [34]:
status_counts = df_merged["status"].value_counts()

num_ok = status_counts.get("ok", 0)
num_dead = status_counts.get("dead", 0)
num_error = status_counts.get("error", 0)
num_total = len(status_counts)
print( "Ok ",num_ok) 
print( "Dead ",num_dead)
print( "Error ",num_error ) 
print( "Total ",num_total )

Ok  41058
Dead  4474
Error  14380
Total  3


In [35]:
df_merged["reason"].value_counts()

reason
HTTPConnectionPool(host='projektwebbar.lansstyrelsen.se', port=80): Max retries exceeded with url: /viss/Sv/detta-beskrivs-i-viss/statusklassning/Pages/default.aspx (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x30fb09bb0>, 'Connection to projektwebbar.lansstyrelsen.se timed out. (connect timeout=15)'))                           6954
HTTPConnectionPool(host='projektwebbar.lansstyrelsen.se', port=80): Max retries exceeded with url: /viss/Sv/detta-beskrivs-i-viss/miljoproblem-och-paverkan/miljoproblem/Pages/default.aspx (Caused by ConnectTimeoutError(<urllib3.connection.HTTPConnection object at 0x31fbfe210>, 'Connection to projektwebbar.lansstyrelsen.se timed out. (connect timeout=15)'))    6954
HTTP 404                                                                                                                                                                                                                                                           

In [36]:
from datetime import date
import os

# Sätt datum
today = date.today().strftime("%Y_%m_%d")

# Se till att katalogen finns
os.makedirs("resultsLanstyrelsen", exist_ok=True)

# Bygg filnamn
outfile = f"resultsLanstyrelsen/links_lst_v1_{today}.csv"

# Exportera
df_merged.to_csv(outfile, index=False, encoding="utf-8")

print(f"[OK] Exported {len(df_lst)} rows to {outfile}")


[OK] Exported 59912 rows to resultsLanstyrelsen/links_lst_v1_2026_02_17.csv


In [37]:
lang_stats = (
    df_merged
    .groupby("lang")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        archived_links=("ia_status", lambda s: (s == "available").sum()),
    )
    .reset_index()
)

lang_stats["broken_pct"] = (
    100 * lang_stats["broken_links"] / lang_stats["total_links"]
).round(1)

lang_stats["broken_lost"] = (
    lang_stats["broken_links"] - lang_stats["archived_links"])

top10_langs = (
    lang_stats
    .sort_values("total_links", ascending=False)
    .head(10)
)
top10_langs[
    [
        "lang",
        "total_links",
        "broken_links",
        "broken_pct",
        "archived_links",
        "broken_lost",
    ]
]

,lang,total_links,broken_links,broken_pct,archived_links,broken_lost
32,sv,52900,3139,5.9,779,2360
6,en,2281,372,16.3,106,266
24,no,990,128,12.9,21,107
29,special,934,55,5.9,8,47
5,de,855,294,34.4,36,258
9,fi,838,92,11.0,36,56
4,da,237,116,48.9,71,45
22,nl,106,72,67.9,19,53
7,es,105,51,48.6,29,22
25,pl,91,37,40.7,29,8


In [38]:
## Special till lang Wikidata  
df_merged["lang"] = df_merged["lang"].replace({"special": "Wikidata"})

In [39]:
from urllib.parse import urlparse

df = df_merged.copy()

df["domain"] = df["url"].apply(
    lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
)
domain_stats = (
    df
    .groupby("domain")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        error_links=("status", lambda s: (s == "error").sum()),
    )
    .reset_index()
)
domain_stats["broken_pct"] = (
    100 * domain_stats["broken_links"] / domain_stats["total_links"]
).round(1)

domain_stats["error_pct"] = (
    100 * domain_stats["error_links"] / domain_stats["total_links"]
).round(1)


In [40]:
domain_stats

,domain,total_links,broken_links,error_links,broken_pct,error_pct
0,ab.lst.se,1,0,0,0.0,0.0
1,ac.lst.se,1,0,0,0.0,0.0
2,atgarderivatten.lansstyrelsen.se,1,0,1,0.0,100.0
3,bd.lst.se,10,9,0,90.0,0.0
4,c.lst.se,1,0,0,0.0,0.0
...,...,...,...,...,...,...
77,www5.h.lst.se,4,0,4,0.0,100.0
78,www5.k.lst.se,1,0,1,0.0,100.0
79,www5.n.lst.se,1,0,1,0.0,100.0
80,www5.o.lst.se,16,0,16,0.0,100.0


In [41]:
status_counts = df["status"].value_counts()

num_ok = int(status_counts.get("ok", 0))
num_dead = int(status_counts.get("dead", 0))
num_error = int(status_counts.get("error", 0))
num_total = len(df)

pct_ok = round(100 * num_ok / num_total, 1)
pct_dead = round(100 * num_dead / num_total, 1)
pct_error = round(100 * num_error / num_total, 1)

# Broken links: archived vs lost
num_dead_archived = df[
    (df["status"] == "dead") & (df["ia_status"] == "available")
].shape[0]

num_dead_lost = num_dead - num_dead_archived


In [42]:
top_domains = (
    domain_stats[domain_stats["total_links"] >= 5]
    .sort_values("total_links", ascending=False)
    .head(10)
)
domain_stats_html = "<ul>"
for _, r in top_domains.iterrows():
    domain_stats_html += (
        f"<li><strong>{r['domain']}</strong>: "
        f"{r['broken_links']} / {r['total_links']} broken "
        f"({r['broken_pct']}%)</li>"
    )
domain_stats_html += "</ul>"


In [48]:
from pathlib import Path
from datetime import date, datetime
from urllib.parse import quote
import pandas as pd

import re

def to_fieldname(s):
    s = s.strip().lower()
    s = re.sub(r"\s+", "_", s)          # spaces → underscore
    s = re.sub(r"[^a-z0-9_]", "", s)    # drop special chars
    s = re.sub(r"^[^a-z_]+", "", s)     # no leading digits/invalid
    return s

def is_qid(title):
    return isinstance(title, str) and title.startswith("Q") and title[1:].isdigit()

def save_sortable_html_df_lst(
    df,
    out_dir="resultsLanstyrelsen",
    domains=None,
    issue_url="https://github.com/salgo60/SCB-Wikidata/issues/68",
    reportTitle="Länsstyrelsen"
):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    today = date.today().strftime("%Y_%m_%d")
    reportTitleFileName = to_fieldname(reportTitle)
    out_path = out_dir / f"links_lst_v1_{today}_{reportTitleFileName}.html"
    rerun_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

     # --- Förbered data ---
    df = df.copy()  
    status_counts = df["status"].value_counts()
    df["is_wikidata"] = df["page_title"].apply(is_qid)
    
    # Normalize lang
    df.loc[df["is_wikidata"], "lang"] = "wikidata"
    df["lang"] = df["lang"].replace({"special": None})

    num_ok = int(status_counts.get("ok", 0))
    num_dead = int(status_counts.get("dead", 0))
    num_error = int(status_counts.get("error", 0))
    num_total = len(df)
    
    pct_ok = round(100 * num_ok / num_total, 1)
    pct_dead = round(100 * num_dead / num_total, 1)
    pct_error = round(100 * num_error / num_total, 1)
    
    # Broken links: archived vs lost
    num_dead_archived = df[
        (df["status"] == "dead") & (df["ia_status"] == "available")
    ].shape[0]
    
    num_dead_lost = num_dead - num_dead_archived

    domains = domains or []
    from urllib.parse import urlparse
    
    df["domain"] = df["url"].apply(
        lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
    )

    domain_stats = (
        df
        .groupby("domain")
        .agg(
            total_links=("url", "count"),
            broken_links=("status", lambda s: (s == "dead").sum()),
            error_links=("status", lambda s: (s == "error").sum()),
        )
        .reset_index()
    )
    
    domain_stats["broken_pct"] = (
        100 * domain_stats["broken_links"] / domain_stats["total_links"]
    ).round(1)
    
    domain_stats["error_pct"] = (
        100 * domain_stats["error_links"] / domain_stats["total_links"]
    ).round(1)
    
    domain_stats["problem_pct"] = (
        100 * (domain_stats["broken_links"] + domain_stats["error_links"])
        / domain_stats["total_links"]
    ).round(1)
    

    domain_stats_html = "<ul>"
    for _, r in top_domains.iterrows():
        domain_stats_html += (
            f"<li><strong>{r['domain']}</strong>: "
            f"{r['broken_links']} / {r['total_links']} broken "
            f"({r['broken_pct']}%)</li>"
        )
    domain_stats_html += "</ul>"

    domain_table_html = (
        domain_stats
        .head(20)
        .to_html(
            classes="pivot",
            border=0,
            index=False,
        )
    )
    lang_stats = (
    df
        .groupby("lang")
        .agg(
            total_links=("url", "count"),
            broken_links=("status", lambda s: (s == "dead").sum()),
            error_links=("status", lambda s: (s == "error").sum()),
            broken_archived=("ia_status", lambda s: (s == "available").sum()),
        )
        .reset_index()
    )
    
    lang_stats["broken_lost"] = (
        lang_stats["broken_links"] - lang_stats["broken_archived"]
    )
    
    lang_stats["broken_pct"] = (
        100 * lang_stats["broken_links"] / lang_stats["total_links"]
    ).round(1)
    
    lang_stats["problem_pct"] = (
        100 * (lang_stats["broken_links"] + lang_stats["error_links"])
        / lang_stats["total_links"]
    ).round(1)

    lang_stats = lang_stats.sort_values(
        "broken_links",
        ascending=False
    )
 
    lang_stats_display = lang_stats[
        [
            "lang",
            "total_links",
            "broken_links",
            "broken_archived",
            "broken_lost",
            "broken_pct",
            "problem_pct",
        ]
    ]

    lang_table_html = (
        lang_stats_display
        .head(15)
        .to_html(
            classes="pivot",
            border=0,
            index=False,
        )
    )

    STATUS_ICON = {
        "ok":    ("fa-circle-check", "#2e7d32", "OK"),
        "dead":  ("fa-circle-xmark", "#c62828", "Broken link"),
        "error": ("fa-triangle-exclamation", "#ef6c00", "Request error"),
    }
    
    if "status" in df.columns:
        def render_status(r):
            icon, color, label = STATUS_ICON.get(
                r["status"], ("fa-question-circle", "#757575", "Unknown")
            )
            reason = r.get("reason", "")
            return (
                f'<span class="status-icon" '
                f'data-status="{r["status"]}" '
                f'title="{label}: {reason}" '
                f'style="color:{color}; font-size:14px; cursor:pointer;">'
                f'<i class="fa-solid {icon}"></i>'
                f'</span>'
            )
        df.insert(
            0,
            "status_icon",
            df.apply(render_status, axis=1)
        )

        def render_ia_icon(r):
            if r.get("ia_status") == "available" and r.get("ia_url"):
                return (
                    f'<a href="{r["ia_url"]}" target="_blank" '
                    f'title="Archived copy (Internet Archive)">'
                    f'<i class="fa-solid fa-box-archive" '
                    f'style="color:#1565c0;"></i>'
                    f'</a>'
                )
            return ""
        
        df.insert(
            1,
            "archive",
            df.apply(render_ia_icon, axis=1)
        )


    # Wikipedia: ikon + titel (byggd från lang + page_title)
    if {"lang", "page_title"}.issubset(df.columns):
    
        def render_page_link(r):
            title = r["page_title"]
    
            # --- Wikidata item ---
            if r.get("is_wikidata"):
                label = r.get("wd_label_sv")
                link_text = label if pd.notna(label) else title
    
                return (
                    f'<a href="https://www.wikidata.org/wiki/{title}" '
                    f'target="_blank" title="Wikidata">'
                    f'<i class="fa-solid fa-database" style="margin-right:6px;"></i>'
                    f'{link_text}</a>'
                )
    
            # --- Wikipedia article ---
            lang = r.get("lang")
            if pd.notna(lang) and pd.notna(title):
                return (
                    f'<a href="https://{lang}.wikipedia.org/wiki/{quote(str(title))}" '
                    f'target="_blank" title="Wikipedia ({lang})">'
                    f'<i class="fa-brands fa-wikipedia-w" style="margin-right:6px;"></i>'
                    f'{title}</a>'
                )
    
            return title

        df["page_title"] = df.apply(render_page_link, axis=1)

    # Externa länkar
    for col in ["Wikipedia-länk", "Extern länk", "url"]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: f'<a href="{x}" target="_blank">{x}</a>' if pd.notna(x) else ""
            )

    # --- HTML-tabell ---
    html_table = df.to_html(
        classes="pivot",
        border=0,
        escape=False,  # krävs för HTML-länkar
        index=False,
    )

    # --- CSS ---
    css = """
    <style>
      body {
        font-family: Arial, sans-serif;
        margin: 20px;
      }
      table.pivot {
        border-collapse: collapse;
        width: 100%;
        font-size: 12px;
      }
      table.pivot th, table.pivot td {
        border: 1px solid #999;
        padding: 6px 8px;
        text-align: left;
        vertical-align: top;
        white-space: normal;
      }
      table.pivot th {
        cursor: pointer;
        background: #f2f2f2;
      }
      table.pivot th:hover {
        background: #e2e2e2;
      }
      table.pivot thead th {
        position: sticky;
        top: 0;
        background: #f2f2f2;
        z-index: 2;
      }
      table.pivot th::after {
        content: "";
        float: right;
        opacity: 0.4;
      }
      table.pivot th.sorted-asc::after {
        content: " ▲";
      }
      table.pivot th.sorted-desc::after {
        content: " ▼";
      }
      /* Row coloring by status */
      table.pivot tr[data-status="dead"] {
         background-color: #fdecea;  /* light red */
      }
      table.pivot tr[data-status="dead"] td:nth-child(2) i {
          color: #1565c0;
        }

      table.pivot tr[data-status="error"] {
          background-color: #fff4e5;  /* light orange */
      }

      table.pivot td a {
        color: #0645ad;
        text-decoration: none;
      }
      table.pivot td a:hover {
        text-decoration: underline;
      }
      .meta {
        background: #f8f8f8;
        border: 1px solid #ccc;
        padding: 12px;
        margin-bottom: 20px;
        font-size: 13px;
      }
      .meta h2 {
        margin-top: 0;
      }
    </style>
    """

    # --- JavaScript (sortering) ---
    js = """
    <script>
    document.addEventListener('DOMContentLoaded', () => {
        // Propagate status from first cell to row
        document.querySelectorAll("table.pivot tbody tr").forEach(row => {
            const statusCell = row.querySelector(".status-icon");
            if (statusCell) {
                row.dataset.status = statusCell.dataset.status;
            }
        });
        let showOnlyBroken = false;

        document.querySelectorAll(".status-icon").forEach(icon => {
            icon.addEventListener("click", event => {
                event.stopPropagation(); // prevent column sort
                showOnlyBroken = !showOnlyBroken;
        
                document.querySelectorAll("table.pivot tbody tr").forEach(row => {
                    if (showOnlyBroken) {
                        row.style.display =
                            row.dataset.status === "dead" ? "" : "none";
                    } else {
                        row.style.display = "";
                    }
                });
            });
        });

        document.querySelectorAll("table.pivot th").forEach((header, colIndex) => {
            header.addEventListener("click", () => {
                const table = header.closest("table");
                const tbody = table.querySelector("tbody");
                const rows = Array.from(tbody.querySelectorAll("tr"));
                const asc = !header.classList.contains("sorted-asc");

                rows.sort((a, b) => {
                    const A = a.children[colIndex].innerText.trim();
                    const B = b.children[colIndex].innerText.trim();
                    const numA = parseFloat(A.replace(",", "."));
                    const numB = parseFloat(B.replace(",", "."));
                    if (!isNaN(numA) && !isNaN(numB)) {
                        return asc ? numA - numB : numB - numA;
                    }
                    return asc ? A.localeCompare(B) : B.localeCompare(A);
                });

                table.querySelectorAll("th").forEach(th =>
                    th.classList.remove("sorted-asc", "sorted-desc")
                );
                header.classList.add(asc ? "sorted-asc" : "sorted-desc");
                rows.forEach(row => tbody.appendChild(row));
            });
        });
    });
    </script>
    """
    status_counts = df["status"].value_counts()
    
    num_ok = status_counts.get("ok", 0)
    num_dead = status_counts.get("dead", 0)
    num_error = status_counts.get("error", 0)
    num_total = len(df)

    # --- Metadata ---
    meta_html = f"""
    <div class="meta">
      <h2>Summary</h2>
    
      <p><strong>Rerun:</strong> {rerun_ts}</p>
      <p><strong>Script:</strong>
         <a href="{SCRIPT_URL}" target="_blank">{SCRIPT_NAME}</a>
      </p>
    
      <p>
        <strong>Links checked:</strong> {num_total}<br>
        <strong style="color:#2e7d32;">OK:</strong> {num_ok} ({pct_ok}%)<br>
        <strong style="color:#c62828;">Broken:</strong> {num_dead} ({pct_dead}%)<br>
        &nbsp;&nbsp;↳ Archived: {num_dead_archived}<br>
        &nbsp;&nbsp;↳ Lost: {num_dead_lost}<br>
        <strong style="color:#ef6c00;">Errors:</strong> {num_error} ({pct_error}%)
      </p>
      <p><strong>Issue:</strong>
         <a href="{issue_url}" target="_blank">{issue_url.split("/")[-1]}</a>
      </p>
    
      <p><strong>Datakällor:</strong><br>
         Wikidata<br>
         MediaWiki API – exturlusage
      </p>
    
      <h2>Domains with broken links</h2>
      <p>Top domains ranked by broken-link impact.</p>
      {domain_table_html}
    </div>
    """


    # --- Slutlig HTML ---
    html = f"""
    <html>
    <head>
      <meta charset="utf-8">
      <title>{reportTitle} links in Wikipedia</title>
      <link rel="stylesheet"
            href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.5.1/css/all.min.css">
      {css}
    </head>
    <body>
      <h1>Wikipedia → {reportTitle} v1</h1>
      {meta_html}
      <p>Sorterbar tabell. Klicka på kolumnrubriker för sortering.</p>
      {html_table}
      {js}
      <h2>Broken-link summary by Wikipedia language</h2>
     <p>
       Languages ranked by broken-link impact (broken + error links).
     </p>
    {lang_table_html}
    </body>
    </html>
    """

    out_path.write_text(html, encoding="utf-8")
    print(f"✅ HTML skapad: {out_path}")


In [49]:

save_sortable_html_df_lst(df_merged)


✅ HTML skapad: resultsLanstyrelsen/links_lst_v1_2026_02_17_lnsstyrelsen.html


In [50]:
save_sortable_html_df_lst(df_naturreservat,reportTitle="Artiklar naturreservat")

✅ HTML skapad: resultsLanstyrelsen/links_lst_v1_2026_02_17_artiklar_naturreservat.html


In [52]:
df_naturreservat_not_ok = df_naturreservat[df_naturreservat["status"] != "ok"]
save_sortable_html_df_lst(df_naturreservat_not_ok,reportTitle="Artiklar naturreservat not ok")

✅ HTML skapad: resultsLanstyrelsen/links_lst_v1_2026_02_17_artiklar_naturreservat_not_ok.html


In [53]:
NotInclude = {
    "Diskussion",
    "Användare"
}

In [57]:
pattern = "|".join(NotInclude)

df_naturreservat_not_ok_Articles = df_naturreservat_not_ok[~df_naturreservat_not_ok["page_title"].str.contains(pattern, na=False)]
save_sortable_html_df_lst(df_naturreservat_not_ok_Articles,reportTitle="Artiklar naturreservat not ok 2")

✅ HTML skapad: resultsLanstyrelsen/links_lst_v1_2026_02_17_artiklar_naturreservat_not_ok_2.html


In [46]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))


Date: 2026-02-17 09:48:20
Total time elapsed: 56 minutes 56.35 seconds
